In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm

### Importing Dataframes

In [2]:
def load_dataframes():
    dataframes = {}

    for file in os.listdir('../dataframes/data_original'):
        if file[-3:] == 'csv':
            df = joblib.load(os.path.join('../dataframes/data_original', file))
            dataframes[file[:-4]] = df
    return dataframes

In [3]:
dataframes = load_dataframes()

In [4]:
age = dataframes['final_age']
rent = dataframes['final_rent']
migr = dataframes['final_migr']

area_pr = dataframes['area_pr']
area_lor = dataframes['area_lor']

In [5]:
# excluding years 2007 and 2008 for age, migr and rent due to incomplete and inconsistent data

age = age[age['year']>=2009].copy()
rent = rent[rent['year']>=2009].copy()
migr = migr[migr['year']>=2009].copy()

### Merging Dataframes

In [6]:
df = age.merge(migr, on=['LOR', 'PR', 'year'], how='outer')\
        .merge(rent, on=['LOR', 'PR', 'year'], how='outer')

### Creating remaining features

In [7]:
df['DEU'] = df['total'] - df['total_foreigners']

### Prognoseraum (PR)

#### Percentage

In [8]:
def percentage(df, total_col, col_name):
    df['_'.join([col_name, 'perc'])] = df[col_name] / df[total_col]
    #df.drop(col_name, axis=1, inplace=True)
    return df

In [9]:
cols_perc = ['below_6', '6-15', '15-18', '18-27', '27-45', '45-55', '55-65',
        '65_above', 'female', 'boomers', 'gen_x', 'millenials', 'children',
        'EU', 'FRA', 'GRC', 'ITA', 'AUT', 'ESP',
        'UK', 'POL', 'BGR', 'ROU', 'HRV', 'former_JUGO', 'BIH', 'SRB',
        'former_UDSSR', 'RUS', 'UKR', 'KAZ', 'Islamic_countries', 'TUR', 'IRN',
        'LBN', 'SYR', 'VNM', 'USA', 'not_identified', 'DEU', 'EU_other',
        'JUGO_other', 'UDSSR_other', 'Islamic_other', 'other', 'Arabic_countries']

In [10]:
df_pr = df.copy()

for col in cols_perc:
    percentage(df_pr, 'total', col)

#### Percentage Change to Prior Year

In [11]:
cols = [c for c in df_pr.columns if c not in ['Bezirk', 'Prognoseraum', 'Bezirksregion', 'Planungsraum', 'LOR', 'PR',
                                                'year', 'PLZ', 'Name', 'Progenoseraumname', 'Bezirksname', 'Kiezname']]

for col in tqdm(cols):
    for p in df_pr['PR'].unique():
        current_year = df_pr.loc[df_pr['PR']==p, col]
        prior_year = df_pr.loc[df_pr['PR']==p, col].shift(1)
        df_pr.loc[df_pr['PR']==p, f'{col}_cng'] = (current_year - prior_year) / prior_year
    #replace inf values with the max of the column
    df_pr[f'{col}_cng'].replace(np.inf, sorted(df_pr[f'{col}_cng'].unique())[-2], inplace=True)

100%|██████████| 103/103 [03:25<00:00,  2.00s/it]


#### Merging with area_pr

In [12]:
df_pr = df_pr.merge(area_pr, on=['PR', 'LOR'])

#### Remaining features

In [13]:
df_pr['pop_per_sqm'] = df_pr['total'] / df_pr['area_sqkm']
df_pr['total_foreigners_perc'] = df_pr['total_foreigners'] / df_pr['total']

In [14]:
joblib.dump(df_pr, './data_app/final_pr.csv')

['./data_app/final_pr.csv']

### Bezirksregion (LOR)

#### Grouping to LOR level

In [15]:
aggs = {
    'total': 'sum', 'below_6': 'sum', '6-15': 'sum', '15-18': 'sum', '18-27': 'sum', '27-45': 'sum', '45-55': 'sum',
    '55-65': 'sum', '65_above': 'sum', 'female': 'sum', 'boomers': 'sum', 'gen_x': 'sum', 'millenials': 'sum',
    'children': 'sum', 'total_foreigners': 'sum', 'EU': 'sum', 'POL': 'sum', 'ITA': 'sum', 'former_UDSSR': 'sum',
    'RUS': 'sum', 'UKR': 'sum', 'KAZ': 'sum', 'former_JUGO': 'sum', 'TUR': 'sum', 'Arabic_countries': 'sum', 'VNM': 'sum',
    'USA': 'sum', 'FRA': 'sum', 'GRC': 'sum', 'AUT': 'sum', 'UK': 'sum', 'BGR': 'sum', 'BIH': 'sum', 'HRV': 'sum', 'SRB': 'sum',
    'Islamic_countries': 'sum', 'IRN': 'sum', 'LBN': 'sum', 'not_identified': 'sum', 'ESP': 'sum', 'ROU': 'sum',
    'SYR': 'sum', 'EU_other': 'sum', 'JUGO_other': 'sum', 'UDSSR_other': 'sum', 'Islamic_other': 'sum',
    'other': 'sum', 'DEU': 'sum',
    
    'apt_size': 'mean', 'sqm_price_all': 'mean', 'rent_cold': 'mean', 'rent_warm': 'mean',
    'net_income': 'mean', 'rent_cold_fraction': 'mean', 'rent_warm_fraction': 'mean',
    'sqm_price_lower': 'mean', 'sqm_price_upper': 'mean'
}

In [16]:
df_lor = df.groupby(['LOR', 'year'], as_index=False).agg(aggs)

#### Percentage

In [17]:
for col in cols_perc:
    percentage(df_lor, 'total', col)

#### Percentage Change to Prior Year

In [18]:
cols = [c for c in df_lor.columns if c not in ['Bezirk', 'Prognoseraum', 'Bezirksregion', 'LOR',
                                               'year', 'PLZ', 'Name', 'Progenoseraumname', 'Bezirksname']]

for col in tqdm(cols):
    for l in df_lor['LOR'].unique():
        current_year = df_lor.loc[df_lor['LOR']==l, col]
        prior_year = df_lor.loc[df_lor['LOR']==l, col].shift(1)
        df_lor.loc[df_lor['LOR']==l, f'{col}_cng'] = (current_year - prior_year) / prior_year
    #replace inf values with the max of the column
    df_lor[f'{col}_cng'].replace(np.inf, sorted(df_lor[f'{col}_cng'].unique())[-2], inplace=True)

100%|██████████| 103/103 [00:54<00:00,  1.88it/s]


#### Merging with area_lor

In [19]:
df_lor = df_lor.merge(area_lor, on=['LOR'])

#### Remaining features

In [20]:
df_lor['pop_per_sqm'] = df_lor['total'] / df_lor['area_sqkm']
df_lor['total_foreigners_perc'] = df_lor['total_foreigners'] / df_lor['total']

In [21]:
joblib.dump(df_lor, './data_app/final_lor.csv')

['./data_app/final_lor.csv']